In [1]:
# Textbook: Principles and Practices of Machine Learning
# Alcohol dataset
# Author: Zhe Chen (ml_iot@163.com), 2021

import pandas as pd
import numpy as np

# load dataset
df = pd.read_csv('alcohol_dataset.csv')
# show a few rows to confirm load
df.head()

,ALCOHOL,TEMP_AMB,TEMP_FAC_MAX,TEMP_FAC_MIN,EYES,LABEL
0,0.2,27.94,27.23,11.75,3.53553,0
1,0.2,27.83,27.26,12.62,3.53553,0
2,0.2,27.94,27.49,12.09,3.53553,0
3,0.2,28.17,26.70,13.01,3.53553,0
4,0.2,28.11,27.15,13.26,3.53553,0


In [2]:
# Prepare features X and labels y (assume label in last column)
# 这是一个注释，说明了这段代码的主要目的：从DataFrame中准备特征矩阵X和标签向量y，并假设标签位于最后一列。

X = df.iloc[:, :-1].values
# 从DataFrame 'df' 中提取特征数据。
# df.iloc: 使用基于整数位置的索引来选择行和列。
# [:, :-1]:
#   - : 表示选择所有行。
#   - :-1 表示选择从第一列到倒数第二列（不包括最后一列）。
# .values: 将选择的数据转换为NumPy数组。
# 这样，X 就包含了DataFrame中除最后一列之外的所有列作为特征。

y = df.iloc[:, -1].values
# 从DataFrame 'df' 中提取标签数据。
# df.iloc: 同样使用基于整数位置的索引。
# [:, -1]:
#   - : 表示选择所有行。
#   - -1 表示选择最后一列。
# .values: 将选择的数据转换为NumPy数组。
# 这样，y 就包含了DataFrame中的最后一列作为标签。

# replace label 0 with -1
# 这是一个注释，说明接下来要进行标签转换：将标签0替换为-1。
# 这在某些机器学习算法（特别是某些SVM实现）中是常见的做法，因为它们可能更倾向于使用-1和1来表示二分类的两个类别。

y = np.where(y == 0, -1, y)
# 使用NumPy的where函数对标签y进行条件替换。
# np.where(condition, x, y):
#   - condition: y == 0，即当y中的元素等于0时。
#   - x: -1，如果条件为真，则将该元素替换为-1。
#   - y: y，如果条件为假（即元素不等于0），则保持原值不变。
# 这行代码将y中所有值为0的元素替换为-1，而其他元素（例如1）保持不变。

print('Class distribution:', np.unique(y, return_counts=True))
# 打印处理后的标签y的类别分布。
# np.unique(y, return_counts=True):
#   - np.unique(y): 返回y中所有唯一的元素（即类别）。
#   - return_counts=True: 同时返回每个唯一元素出现的次数。
# 这个输出会显示当前标签y中有哪些类别（例如-1和1），以及每个类别有多少个样本。
# 这对于检查标签转换是否成功以及了解数据集的类别平衡情况非常有用。


Class distribution: (array([-1,  1]), array([213, 171]))


In [3]:
# Split data and standardize features
# 这是一个注释，说明了这段代码的主要目的：将数据划分为训练集和测试集，并对特征进行标准化处理。

from sklearn.model_selection import train_test_split
# 从scikit-learn库的model_selection模块中导入train_test_split函数。
# 这个函数用于将数据集随机划分为训练子集和测试子集。

from sklearn.preprocessing import StandardScaler
# 从scikit-learn库的preprocessing模块中导入StandardScaler类。
# StandardScaler用于对特征进行标准化（也称为Z-score标准化），使其均值为0，标准差为1。

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# 使用train_test_split函数将原始数据集X（特征）和y（目标变量/标签）划分为训练集和测试集。
# X: 原始特征数据集。
# y: 原始目标变量（标签）数据集。
# test_size=0.3: 指定测试集占总数据集的比例为30%（即训练集占70%）。
# random_state=42: 设置随机种子为42。这确保每次运行代码时，数据的划分结果都是一致的，便于结果复现。
# stratify=y: 这是一个非常重要的参数，尤其是在处理类别不平衡的数据集时。
# 它确保训练集和测试集中目标变量y的类别比例与原始数据集X中y的类别比例保持一致。
# 例如，如果y中有10%的正样本，那么训练集和测试集中也会有大约10%的正样本。

scaler = StandardScaler()
# 初始化一个StandardScaler对象。这个对象将用来计算训练数据的均值和标准差，并用它们来转换数据。

X_train = scaler.fit_transform(X_train)
# 对训练集X_train进行标准化。
# scaler.fit_transform() 方法做了两件事：
# 1. fit(): 计算X_train中每个特征的均值（mean）和标准差（std）。
# 2. transform(): 使用计算出的均值和标准差来转换X_train，使其每个特征的均值为0，标准差为1。
# 重要的是，均值和标准差是从训练数据中学习的，以避免数据泄露。

X_test = scaler.transform(X_test)
# 对测试集X_test进行标准化。
# 注意这里只使用了 scaler.transform()，而不是 fit_transform()。
# 这意味着测试集是使用**训练集**的均值和标准差进行转换的。
# 这样做是为了模拟真实世界中，模型在训练时只见过训练数据，而在预测新数据时，新数据（测试集）应该使用训练时学到的统计量进行预处理，以保证数据处理的一致性，并避免测试集信息泄露到训练过程中。

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
# 打印处理后的训练集和测试集的形状（行数和列数）。
# 这有助于确认数据划分和处理是否正确，例如，确认训练集和测试集的样本数量以及特征数量。


Train shape: (268, 5) Test shape: (116, 5)


In [6]:
# Train and evaluate SVMs with different kernels and C values
# 这是一个注释，说明了这段代码的主要目的：使用不同的核函数和C值来训练和评估支持向量机（SVM）。

from sklearn import svm
# 从scikit-learn库中导入svm模块，该模块包含了支持向量机模型（如SVC）。

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# 从scikit-learn库的metrics模块中导入常用的分类评估指标：
# accuracy_score: 准确率
# precision_score: 精确率
# recall_score: 召回率
# f1_score: F1分数（精确率和召回率的调和平均值）
# confusion_matrix: 混淆矩阵

import pandas as pd
# 导入pandas库，用于数据处理和创建DataFrame，方便存储和展示实验结果。

results = []
# 初始化一个空列表，用于存储每次SVM模型训练和评估的结果。
# 每个结果将是一个字典，包含模型的超参数和各项评估指标。

kernels = ['linear', 'poly', 'rbf']
# 定义一个列表，包含要尝试的不同核函数类型：
# 'linear': 线性核函数
# 'poly': 多项式核函数
# 'rbf': 径向基函数核（高斯核）

Cs = [0.1, 1, 10, 100, 1000]
# 定义一个列表，包含要尝试的不同C值（惩罚参数）。
# C值越大，对错分样本的惩罚越大，模型越倾向于过拟合；C值越小，模型越倾向于欠拟合。

for kernel in kernels:
    # 外层循环：遍历所有定义的核函数类型。
    for C in Cs:
        # 内层循环：对于每种核函数，遍历所有定义的C值。
        
        if kernel == 'poly':
            # 如果当前核函数是多项式核（'poly'），则需要额外指定degree参数。
            clf = svm.SVC(kernel=kernel, C=C, degree=3, gamma='scale')
            # 初始化一个SVC分类器。
            # kernel=kernel: 指定当前循环的核函数。
            # C=C: 指定当前循环的惩罚参数C。
            # degree=3: 对于多项式核，指定多项式的阶数为3。
            # gamma='scale': gamma参数用于RBF、poly和sigmoid核，'scale'表示使用1 / (n_features * X.var())作为gamma值。
        else:
            # 如果当前核函数不是多项式核，则不需要degree参数。
            clf = svm.SVC(kernel=kernel, C=C, gamma='scale')
            # 初始化一个SVC分类器，不指定degree参数。
            
        clf.fit(X_train, y_train)
        # 使用训练数据X_train和对应的标签y_train来训练当前的SVM模型。

        y_pred = clf.predict(X_test)
        # 使用训练好的模型对测试数据X_test进行预测，得到预测标签y_pred。

        acc = accuracy_score(y_test, y_pred)
        # 计算模型的准确率（accuracy），即正确预测的样本比例。
        prec = precision_score(y_test, y_pred, pos_label=1)
        # 计算模型的精确率（precision），pos_label=1表示将1视为正类别。
        # 精确率 = 真阳性 / (真阳性 + 假阳性)
        rec = recall_score(y_test, y_pred, pos_label=1)
        # 计算模型的召回率（recall），pos_label=1表示将1视为正类别。
        # 召回率 = 真阳性 / (真阳性 + 假阴性)
        f1 = f1_score(y_test, y_pred, pos_label=1)
        # 计算模型的F1分数，是精确率和召回率的调和平均值。
        cm = confusion_matrix(y_test, y_pred)
        # 计算混淆矩阵，用于更详细地分析模型的分类性能。

        n_support = clf.n_support_.sum() if hasattr(clf, 'n_support_') else None
        # 获取支持向量的数量。
        # clf.n_support_ 是一个数组，表示每个类别的支持向量数量。sum()将其加起来得到总数。
        # hasattr(clf, 'n_support_') 检查模型是否有n_support_属性，以防某些情况下（如模型训练失败）该属性不存在。

        results.append({'kernel': kernel, 'C': C, 'accuracy': acc, 'precision': prec, 'rec': rec, 'f1': f1, 'n_support': n_support, 'confusion_matrix': cm})
        # 将当前模型的超参数（kernel, C）和所有评估指标（accuracy, precision, recall, f1, n_support, confusion_matrix）
        # 作为一个字典添加到results列表中。

res_df = pd.DataFrame(results).sort_values(by='accuracy', ascending=False).reset_index(drop=True)
# 将results列表转换为一个pandas DataFrame。
# .sort_values(by='accuracy', ascending=False): 按准确率（accuracy）降序排序，以便于查看表现最好的模型。
# .reset_index(drop=True): 重置DataFrame的索引，并丢弃旧的索引。

res_df
# 打印或显示最终的DataFrame，其中包含了所有实验组合的性能结果。


,kernel,C,accuracy,precision,rec,f1,n_support,confusion_matrix
0,linear,0.1,0.991379,1.000000,0.980769,0.990291,51,"[[64, 0], [1, 51]]"
1,linear,1.0,0.991379,1.000000,0.980769,0.990291,20,"[[64, 0], [1, 51]]"
2,linear,10.0,0.991379,1.000000,0.980769,0.990291,15,"[[64, 0], [1, 51]]"
3,linear,100.0,0.991379,1.000000,0.980769,0.990291,14,"[[64, 0], [1, 51]]"
4,linear,1000.0,0.991379,1.000000,0.980769,0.990291,14,"[[64, 0], [1, 51]]"
5,poly,10.0,0.991379,1.000000,0.980769,0.990291,43,"[[64, 0], [1, 51]]"
6,rbf,0.1,0.991379,1.000000,0.980769,0.990291,148,"[[64, 0], [1, 51]]"
7,poly,1.0,0.982759,1.000000,0.961538,0.980392,90,"[[64, 0], [2, 50]]"
8,poly,100.0,0.982759,0.962963,1.000000,0.981132,27,"[[62, 2], [0, 52]]"
9,rbf,1.0,0.982759,1.000000,0.961538,0.980392,60,"[[64, 0], [2, 50]]"


In [5]:
# Show best model details and support vector info
# 这是一个注释，说明了这段代码的主要目的：展示表现最佳模型的详细信息和支持向量信息。

best = res_df.iloc[0]
# 从之前创建的DataFrame (res_df) 中获取第一行数据。
# 由于res_df是按准确率降序排序的，所以iloc[0]对应的是准确率最高的模型。
# 'best' 现在是一个Series对象，包含了最佳模型的超参数和评估指标。

print('Best model:', best['kernel'], 'C=', best['C'])
# 打印最佳模型的核函数类型和C值，方便用户快速识别。

best_kernel = best['kernel']
# 提取最佳模型的核函数类型。
best_C = best['C']
# 提取最佳模型的C值。

if best_kernel == 'poly':
    # 重新初始化最佳模型：如果最佳核函数是多项式核，则需要指定degree参数。
    best_clf = svm.SVC(kernel=best_kernel, C=best_C, degree=3, gamma='scale')
    # 使用最佳的核函数、C值和预设的degree=3（与之前实验保持一致）来创建一个新的SVC分类器实例。
else:
    # 如果最佳核函数不是多项式核，则不需要degree参数。
    best_clf = svm.SVC(kernel=best_kernel, C=best_C, gamma='scale')
    # 使用最佳的核函数和C值来创建一个新的SVC分类器实例。

best_clf.fit(X_train, y_train)
# 使用完整的训练数据X_train和y_train来训练这个最佳模型。
# 这一步是必要的，因为'best' Series中只保存了超参数和评估结果，并没有保存训练好的模型实例本身。

y_pred_best = best_clf.predict(X_test)
# 使用训练好的最佳模型对测试数据X_test进行预测，得到预测标签。

print('Accuracy:', accuracy_score(y_test, y_pred_best))
# 重新计算并打印最佳模型在测试集上的准确率。
# 理论上，这个值应该与res_df中best['accuracy']的值非常接近（如果不是完全相同，可能是浮点数精度问题）。
print('Precision:', precision_score(y_test, y_pred_best, pos_label=1))
# 打印最佳模型的精确率。
print('Recall:', recall_score(y_test, y_pred_best, pos_label=1))
# 打印最佳模型的召回率。
print('F1:', f1_score(y_test, y_pred_best, pos_label=1))
# 打印最佳模型的F1分数。

print('Support vectors per class:', best_clf.n_support_)
# 打印每个类别的支持向量数量。
# n_support_ 是一个数组，例如 [num_class_0_sv, num_class_1_sv]，显示了每个类别有多少个支持向量。
print('Total support vectors:', best_clf.n_support_.sum())
# 打印支持向量的总数量，这是所有类别支持向量数量的总和。
# 支持向量的数量是衡量SVM模型复杂度的重要指标，数量越少通常意味着模型越简洁。

print('Support indices (first 20):', best_clf.support_[:20])
# 打印支持向量的索引。
# support_ 属性存储了训练数据中作为支持向量的样本的索引。
# 这里只打印了前20个索引，以避免输出过长。这些索引可以用来回溯查看哪些训练样本是支持向量。


Best model: linear C= 0.1
Accuracy: 0.9913793103448276
Precision: 1.0
Recall: 0.9807692307692307
F1: 0.9902912621359223
Support vectors per class: [25 26]
Total support vectors: 51
Support indices (first 20): [  7   8  10  17  22  56  73  84  87 111 125 141 162 163 165 180 181 210
 226 228]
